# La Argentina desigual: territorio, pobreza y educación en el Censo 2022

---

El **Censo Nacional de Población, Hogares y Viviendas 2022** (CPV2022) es la radiografía estadística más completa del país. Con el paquete `censoargentino` accedés a esos datos en segundos desde Python, sin descargas masivas: DuckDB filtra y trae solo lo que necesitás.

Esta notebook propone un recorrido por la **desigualdad territorial** usando datos reales del censo. No es un tutorial de funciones — es un análisis.

**Preguntas que vamos a responder:**
1. ¿Qué tan desigual es la distribución del NBI entre provincias?
2. ¿Cuáles son los extremos y qué regiones concentran la pobreza estructural?
3. ¿Cuánto varía la situación *dentro* de una provincia, a nivel departamental?
4. ¿Qué dicen las distintas dimensiones del IPMH sobre la privación material?
5. ¿Cómo se distribuye el nivel educativo y qué tan ancha es la brecha?

```
INDEC (REDATAM) → procesamiento → Hugging Face → censoargentino → este análisis
```

In [ ]:
import censoargentino as censo
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
})

# Mapa de regiones geográficas (para colorear gráficos)
REGION = {
    "Buenos Aires": "Pampeana/AMBA", "Caba": "Pampeana/AMBA",
    "Córdoba": "Pampeana/AMBA",     "Santa Fe": "Pampeana/AMBA",
    "Entre Ríos": "Pampeana/AMBA",  "La Pampa": "Pampeana/AMBA",
    "Mendoza": "Cuyo", "San Juan": "Cuyo", "San Luis": "Cuyo",
    "Salta": "NOA", "Jujuy": "NOA", "Tucumán": "NOA",
    "Catamarca": "NOA", "La Rioja": "NOA", "Santiago del Estero": "NOA",
    "Chaco": "NEA", "Corrientes": "NEA", "Formosa": "NEA", "Misiones": "NEA",
    "Neuquén": "Patagonia", "Río Negro": "Patagonia",
    "Chubut": "Patagonia", "Santa Cruz": "Patagonia", "Tierra del Fuego": "Patagonia",
}
COL_REGION = {
    "Pampeana/AMBA": "#2980b9",
    "Cuyo":          "#8e44ad",
    "NOA":           "#e67e22",
    "NEA":           "#27ae60",
    "Patagonia":     "#16a085",
}

print("Listo.")

---
## 0. El catálogo: ¿qué hay disponible?

El censo 2022 tiene tres entidades principales: **PERSONA**, **HOGAR** y **VIVIENDA**. `censo.variables()` devuelve el catálogo completo.

In [ ]:
vars_df = censo.variables()
resumen = vars_df[vars_df["entidad"].isin(["PERSONA", "HOGAR", "VIVIENDA"])]
print(f"Variables disponibles: {len(resumen)} (PERSONA / HOGAR / VIVIENDA)")
print(resumen.groupby("entidad").size().rename("n").to_string())

---
## 1. ¿Cuántos somos y cómo nos distribuimos?

El CPV2022 registró **45,6 millones de personas**. La distribución por sexo es el dato más básico — y ya muestra una leve mayoría femenina consistente con la esperanza de vida diferencial.

`censo.tabla()` descarga y resume en una sola llamada.

In [ ]:
sexo = censo.tabla("PERSONA_P02")
sexo

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.8))
colores_s = ["#e07b8a", "#5b8ec4"]
ax.bar(sexo["categoria"], sexo["N"] / 1_000_000,
       color=colores_s, width=0.45, alpha=0.9)

for i, (n, pct) in enumerate(zip(sexo["N"], sexo["%"])):
    ax.text(i, n / 1_000_000 + 0.15,
            f"{n/1_000_000:.1f}M\n({pct}%)",
            ha="center", fontsize=11, fontweight="bold")

ax.set_ylim(0, sexo["N"].max() / 1_000_000 * 1.3)
ax.set_ylabel("Millones de personas")
ax.set_title("Distribución por sexo — Argentina, Censo 2022")
ax.yaxis.set_visible(False)
for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout()
plt.show()

---
## 2. El mapa de la desigualdad: NBI por provincia

El **NBI (Necesidades Básicas Insatisfechas)** identifica hogares con carencias en vivienda, saneamiento, hacinamiento, educación o capacidad de subsistencia. Es el indicador clásico de pobreza estructural.

`censo.comparar()` devuelve una tabla pivot: provincias en filas, categorías en columnas, porcentajes como valores. Una línea de código.

> **¿Qué esperamos ver?** Una fuerte concentración del NBI en el NEA (Chaco, Formosa, Corrientes, Misiones) y el NOA (Salta, Jujuy, Santiago del Estero), y valores bajos en la región pampeana y la Patagonia.

In [ ]:
nbi_prov = censo.comparar("HOGAR_NBI_TOT")
col_nbi = [c for c in nbi_prov.columns if c not in ("No", "Total")][0]  # columna NBI positivo
nbi_prov.sort_values(col_nbi, ascending=False)

In [ ]:
plot_df = nbi_prov[[col_nbi, "Total"]].sort_values(col_nbi)
colores_prov = [COL_REGION.get(REGION.get(p, ""), "#aaaaaa") for p in plot_df.index]
media = plot_df[col_nbi].mean()

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(range(len(plot_df)), plot_df[col_nbi],
        color=colores_prov, alpha=0.85, height=0.72)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df.index, fontsize=9)

for i, (prov, row) in enumerate(plot_df.iterrows()):
    ax.text(row[col_nbi] + 0.15, i, f"{row[col_nbi]:.1f}%", va="center", fontsize=8)

ax.axvline(media, color="black", ls="--", lw=1, alpha=0.5)
ax.text(media + 0.15, len(plot_df) - 0.6, f"Media: {media:.1f}%",
        fontsize=8, va="top", color="#444")

leyenda = [mpatches.Patch(color=c, label=r) for r, c in COL_REGION.items()]
ax.legend(handles=leyenda, title="Región", loc="lower right", fontsize=8)
ax.set_xlabel("% hogares con NBI")
ax.set_title("Hogares con Necesidades Básicas Insatisfechas (NBI)\npor provincia — Argentina, Censo 2022")
ax.set_xlim(0, plot_df[col_nbi].max() * 1.22)
plt.tight_layout()
plt.show()

**Lo que muestra el gráfico:**

- La brecha entre la provincia con menor NBI (**La Pampa, ~2.6%**) y la de mayor (**Tierra del Fuego, ~13%**, o provincias del NEA/NOA) es de casi 5x.
- Todas las provincias del **NEA** (verde) superan la media nacional.
- La **Patagonia** muestra valores moderados a pesar de ser una región de alta migración interna con asentamientos precarios en su periferia urbana.
- El **AMBA** concentra el mayor volumen absoluto de hogares con NBI (Buenos Aires tiene millones de hogares aunque el porcentaje sea medio).

---
## 3. Los extremos: las dos Argentinas

Los promedios provinciales esconden situaciones extremas. ¿Qué tan diferentes son las provincias en los extremos del ranking?

In [ ]:
top5 = nbi_prov[[col_nbi, "Total"]].sort_values(col_nbi, ascending=False).head(5)
bot5 = nbi_prov[[col_nbi, "Total"]].sort_values(col_nbi).head(5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
escala_max = nbi_prov[col_nbi].max() * 1.25

# Peor 5
t = top5.sort_values(col_nbi)
ax1.barh(range(len(t)), t[col_nbi], color="#c0392b", alpha=0.85)
ax1.set_yticks(range(len(t)))
ax1.set_yticklabels(t.index, fontsize=10)
for i, v in enumerate(t[col_nbi]):
    ax1.text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=10, fontweight="bold")
ax1.set_xlim(0, escala_max)
ax1.set_xlabel("% hogares con NBI")
ax1.set_title("Provincias con mayor NBI", color="#c0392b")
ax1.axvline(media, color="black", ls="--", lw=0.8, alpha=0.4)

# Mejor 5
b = bot5.sort_values(col_nbi, ascending=False)
ax2.barh(range(len(b)), b[col_nbi], color="#27ae60", alpha=0.85)
ax2.set_yticks(range(len(b)))
ax2.set_yticklabels(b.index, fontsize=10)
for i, v in enumerate(b[col_nbi]):
    ax2.text(v + 0.1, i, f"{v:.1f}%", va="center", fontsize=10, fontweight="bold")
ax2.set_xlim(0, escala_max)
ax2.set_xlabel("% hogares con NBI")
ax2.set_title("Provincias con menor NBI", color="#27ae60")
ax2.axvline(media, color="black", ls="--", lw=0.8, alpha=0.4, label=f"Media: {media:.1f}%")
ax2.legend(fontsize=8)

fig.suptitle("Los extremos del NBI provincial — Argentina, Censo 2022",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Ratio entre extremos
max_nbi = top5[col_nbi].max()
min_nbi = bot5[col_nbi].min()
print(f"Ratio extremos: {max_nbi:.1f}% / {min_nbi:.1f}% = {max_nbi/min_nbi:.1f}x")

---
## 4. Más allá del NBI: las dimensiones de la privación material (IPMH)

El **IPMH (Índice de Privación Material de los Hogares)** es más sofisticado que el NBI: combina dos dimensiones independientes.

| Categoría | Qué significa |
|---|---|
| Sin privación | No presenta carencias en ninguna dimensión |
| Privación de recursos corrientes | Ingresos insuficientes para cubrir necesidades básicas |
| Privación patrimonial | Vivienda o hacinamiento inadecuados |
| Privación convergente | **Ambas** dimensiones simultáneamente — la situación más crítica |

La **privación convergente** es el indicador más severo y el que mejor identifica situaciones de pobreza estructural profunda.

In [ ]:
# Distribución nacional
ipmh_nac = censo.tabla("HOGAR_IPMH")
print("Argentina — distribución del IPMH:")
ipmh_nac

In [ ]:
ipmh_prov = censo.comparar("HOGAR_IPMH")
cols_ipmh = [c for c in ipmh_prov.columns if c != "Total"]
col_sin_priv = cols_ipmh[0]   # "Sin privación"
col_conv = cols_ipmh[-1]      # "Privación Convergente"

# Ordenar por "sin privación" ascendente (peores arriba en barh)
plot_ipmh = ipmh_prov[cols_ipmh].sort_values(col_sin_priv, ascending=True)

colores_ipmh = ["#2ecc71", "#f39c12", "#e67e22", "#c0392b"]
fig, ax = plt.subplots(figsize=(11, 9))
plot_ipmh.plot(kind="barh", stacked=True, ax=ax,
               color=colores_ipmh[:len(cols_ipmh)], alpha=0.88, width=0.72)

ax.set_xlabel("% hogares")
ax.set_title("Índice de Privación Material de los Hogares (IPMH)\npor provincia — Argentina, Censo 2022")
ax.legend(title="Categoría", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
ax.tick_params(axis="y", labelsize=9)
ax.set_xlim(0, 105)
plt.tight_layout()
plt.show()

# Provincias con mayor privación convergente
print(f"\nPrivación convergente — top 5:")
print(ipmh_prov[[col_conv, "Total"]].sort_values(col_conv, ascending=False).head(5).to_string())

**Lectura del gráfico:**

Las provincias con **más verde** (sin privación) a la derecha son las que concentran menor pobreza material. Las de más rojo y naranja son las que acumulan ambas dimensiones de privación.

El ordenamiento del IPMH no siempre coincide exactamente con el del NBI: hay provincias con NBI moderado pero privación de recursos corrientes alta (pobreza por ingresos sin déficit habitacional), y viceversa.

---
## 5. Zoom: la desigualdad *dentro* de una provincia

Los promedios provinciales son útiles para comparar, pero **dentro de cada provincia hay una heterogeneidad enorme**. El departamento capital de Chaco tiene condiciones muy distintas al interior rural.

`censo.comparar(nivel="departamento")` baja un nivel más de granularidad — y ahí es donde se ven las verdaderas brechas.

In [ ]:
# Cambiá esta variable para explorar cualquier provincia
PROVINCIA_ZOOM = "Chaco"

nbi_dpto = censo.comparar("HOGAR_NBI_TOT", nivel="departamento", provincia=PROVINCIA_ZOOM)
col_nbi_d = [c for c in nbi_dpto.columns if c not in ("No", "Total")][0]

# Media provincial (del resultado ya calculado)
try:
    media_prov = nbi_prov.loc[PROVINCIA_ZOOM, col_nbi]
except KeyError:
    media_prov = nbi_dpto[col_nbi_d].mean()

nbi_dpto.sort_values(col_nbi_d, ascending=False)

In [ ]:
plot_d = nbi_dpto[[col_nbi_d]].sort_values(col_nbi_d)

fig, ax = plt.subplots(figsize=(8, max(5, len(plot_d) * 0.44)))
colores_d = ["#c0392b" if v > media_prov else "#e8965a" for v in plot_d[col_nbi_d]]
ax.barh(range(len(plot_d)), plot_d[col_nbi_d],
        color=colores_d, alpha=0.87, height=0.72)
ax.set_yticks(range(len(plot_d)))
ax.set_yticklabels(plot_d.index, fontsize=9)

for i, v in enumerate(plot_d[col_nbi_d]):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=8)

ax.axvline(media_prov, color="black", ls="--", lw=1, alpha=0.65,
           label=f"Media provincial: {media_prov:.1f}%")
ax.set_xlabel("% hogares con NBI")
ax.set_title(f"NBI por departamento — {PROVINCIA_ZOOM}, Censo 2022\n"
             f"rojo = sobre la media provincial ({media_prov:.1f}%)")
ax.legend(fontsize=9)
ax.set_xlim(0, plot_d[col_nbi_d].max() * 1.22)
plt.tight_layout()
plt.show()

rango = plot_d[col_nbi_d].max() - plot_d[col_nbi_d].min()
print(f"Rango interno: {plot_d[col_nbi_d].min():.1f}% a {plot_d[col_nbi_d].max():.1f}% "
      f"(diferencia de {rango:.1f} puntos porcentuales)")

> **Para explorar otra provincia:** cambiá `PROVINCIA_ZOOM` en la celda anterior y volvé a ejecutar. Probá con `"Salta"`, `"Tucumán"` o `"Misiones"`.

La variación **intra-provincial** suele ser tan grande como la variación entre provincias. El departamento con mayor NBI dentro de Chaco puede triplicar al de menor NBI — todo dentro del mismo territorio, a pocas horas de distancia.

---
## 6. La fractura educativa

El nivel de instrucción es tanto un indicador de capital humano como de desigualdad histórica acumulada. La distancia entre el porcentaje de personas **sin instrucción** y con **universitario completo** muestra la brecha educativa de cada provincia.

Consultamos `PERSONA_MNI` (Máximo Nivel de Instrucción) con `comparar()` — obtenemos los porcentajes de las 12 categorías para todas las provincias en una sola llamada.

In [ ]:
mni_prov = censo.comparar("PERSONA_MNI")

# Excluir categoría ignorado
mni_prov = mni_prov[[c for c in mni_prov.columns if "gnor" not in c.lower()]]

# Columnas extremas
col_sin = [c for c in mni_prov.columns if "Sin " in c and c != "Total"][0]
col_uni = [c for c in mni_prov.columns if "Universitario completo" in c][0]
col_sec = [c for c in mni_prov.columns if "Secundario completo" in c][0]

print(f"Extremos analizados: '{col_sin}' vs '{col_uni}'")
mni_prov[[col_sin, col_sec, col_uni, "Total"]].sort_values(col_sin, ascending=False)

In [ ]:
# Ordenar por % sin instrucción descendente
edu_plot = mni_prov[[col_sin, col_uni]].sort_values(col_sin, ascending=False)

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(14, 9), sharey=True)
fig.subplots_adjust(wspace=0.06)
y = range(len(edu_plot))
media_sin = edu_plot[col_sin].mean()
media_uni = edu_plot[col_uni].mean()

# Izquierda: Sin instrucción
cl = ["#c0392b" if v > media_sin else "#e8b4b8" for v in edu_plot[col_sin]]
ax_l.barh(y, edu_plot[col_sin], color=cl, alpha=0.88)
ax_l.set_yticks(y)
ax_l.set_yticklabels(edu_plot.index, fontsize=9)
ax_l.axvline(media_sin, color="black", ls="--", lw=0.8, alpha=0.5)
ax_l.set_xlabel(f"% sin instrucción")
ax_l.set_title(col_sin, fontsize=11, color="#c0392b")
for i, v in enumerate(edu_plot[col_sin]):
    ax_l.text(v + 0.05, i, f"{v:.1f}%", va="center", fontsize=7.5)

# Derecha: Universitario completo
cr = ["#2980b9" if v > media_uni else "#aed6f1" for v in edu_plot[col_uni]]
ax_r.barh(y, edu_plot[col_uni], color=cr, alpha=0.88)
ax_r.axvline(media_uni, color="black", ls="--", lw=0.8, alpha=0.5)
ax_r.set_xlabel(f"% universitario completo")
ax_r.set_title(col_uni, fontsize=11, color="#2980b9")
for i, v in enumerate(edu_plot[col_uni]):
    ax_r.text(v + 0.05, i, f"{v:.1f}%", va="center", fontsize=7.5)

fig.suptitle(
    "Brecha educativa por provincia — Censo 2022\n"
    "(ordenado por % sin instrucción, mayor a menor)",
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.show()

**Lo que muestra el gráfico:**

El ordenamiento por `sin instrucción` suele ser **casi perfectamente inverso** al de `universitario completo`. Las provincias con más analfabetismo funcional tienen menos universitarios graduados — y viceversa. Esto sugiere que la brecha no es solo de nivel educativo, sino de acceso al sistema en todos sus tramos.

**CABA y Córdoba** concentran los valores más altos de graduación universitaria, reflejo de sus sistemas universitarios históricos. Las provincias del NOA y NEA, por el contrario, aparecen consistentemente en el extremo opuesto.

---
## 7. Datos crudos: trabajo directo con los radios censales

Cuando necesitás más control sobre el análisis — filtros específicos, cruce manual de variables, exportación — usás `censo.query()` que devuelve el dataset en formato largo por **radio censal** (~52.000 unidades geográficas en todo el país).

`censo.agregar()` permite resumir ese DataFrame sin volver a hacer una descarga.

In [ ]:
# Datos de NBI para Formosa — a nivel de radio censal
df_formosa = censo.query(variables="HOGAR_NBI_TOT", provincia="Formosa")
print(f"Filas (radios × categorías): {len(df_formosa):,}")
print(f"Radios únicos: {df_formosa['id_geo'].nunique():,}")
df_formosa.head(3)

In [ ]:
# Agregar por departamento — sin nueva descarga
nbi_formosa_dpto = censo.agregar(df_formosa, por="departamento")

# Tabla pivoteada manual para ver el NBI% por departamento
col_si = [c for c in nbi_formosa_dpto["categoria"].unique() if c not in ("No",)][0]
nbi_si = (
    nbi_formosa_dpto[nbi_formosa_dpto["categoria"] == col_si]
    [["etiqueta_departamento", "N", "%"]]
    .rename(columns={"etiqueta_departamento": "departamento", "%": "NBI%"})
    .sort_values("NBI%", ascending=False)
    .reset_index(drop=True)
)
print("NBI por departamento — Formosa (desde datos crudos):")
nbi_si

---
## Conclusiones y próximos pasos

### Lo que encontramos

El Censo 2022 confirma patrones estructurales de larga data:

- **La brecha NBI entre provincias es de 5x** entre los extremos. El NEA y el NOA concentran sistemáticamente los peores indicadores.
- **El IPMH agrega matices**: hay provincias con NBI moderado pero alta privación de recursos corrientes, indicando pobreza por ingresos sin déficit habitacional visible.
- **La heterogeneidad intra-provincial es enorme**: dentro de una misma provincia, el departamento con mayor NBI puede triplicar al de menor. El promedio provincial esconde realidades muy distintas.
- **La brecha educativa sigue el mismo mapa**: las provincias con más analfabetismo funcional son exactamente las que tienen menos universitarios graduados. La desigualdad se acumula en el mismo territorio.

### Variables para seguir explorando

```python
# Acceso a servicios básicos
censo.comparar("VIVIENDA_TIPOVIVG")          # tipo de vivienda por provincia
censo.tabla("VIVIENDA_URP")                  # urbano vs rural nacional
censo.comparar("HOGAR_NBI_SAN")              # NBI saneamiento
censo.comparar("HOGAR_NBI_VIV")              # NBI vivienda

# Demografía y condición de actividad
censo.tabla("PERSONA_EDADGRU")               # grupos de edad nacional
censo.comparar("PERSONA_CONDACT")            # actividad económica

# Zoom departamental en distintas provincias
censo.comparar("HOGAR_NBI_TOT", nivel="departamento", provincia="Misiones")
censo.comparar("PERSONA_MNI",   nivel="departamento", provincia="Córdoba")
censo.comparar("HOGAR_IPMH",    nivel="departamento", provincia="Buenos Aires")

# Datos crudos para análisis propio
df = censo.query(variables="HOGAR_H20CP", provincia="Tucumán")  # hacinamiento
censo.agregar(df, por="departamento")
```

### Recursos

- [Dataset en Hugging Face](https://huggingface.co/datasets/pedroorden/censoargentino)
- [Definiciones de variables — INDEC](https://redatam.indec.gob.ar/redarg/CENSOS/CPV2022/Docs/Redatam_Definiciones_de_la_base_de_datos.pdf)
- [Portal REDATAM online](https://redatam.indec.gob.ar/binarg/RpWebEngine.exe/Portal?BASE=CPV2022&lang=ESP)
- [censoargentino en PyPI](https://pypi.org/project/censoargentino/)